# BQuant Data Integrity Audit

Notebook này dùng để kiểm tra integrity của dữ liệu hiện có trong `data/` và `warehouse/bquant.duckdb`.

Phạm vi kiểm tra hiện tại:
- Inventory của toàn bộ dataset trong `configs/dataset_registry.yaml`
- So khớp giữa DuckDB table, file Parquet và `data_file_manifest`
- Coverage theo symbol cho `daily_ohlcv_10y` và `intraday_ohlcv_15m_60d`
- Rule OHLCV cơ bản: duplicate key, null field, price/volume bất thường, interval 15m lệch nhịp

Notebook này chỉ đọc dữ liệu, không ghi thay đổi.

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any

import duckdb
import pandas as pd
import yaml

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 200)


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "dataset_registry.yaml").exists() and (candidate / "warehouse" / "bquant.duckdb").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repo root from the current working directory.")


REPO_ROOT = find_repo_root()
DB_PATH = REPO_ROOT / "warehouse" / "bquant.duckdb"
REGISTRY_PATH = REPO_ROOT / "configs" / "dataset_registry.yaml"

with REGISTRY_PATH.open("r", encoding="utf-8") as handle:
    DATASET_REGISTRY = (yaml.safe_load(handle) or {}).get("datasets", {})

print(f"Repo root: {REPO_ROOT}")
print(f"DuckDB path: {DB_PATH}")
print(f"Registered datasets: {len(DATASET_REGISTRY)}")


In [ ]:
def resolve_path(path_value: str) -> Path:
    path = Path(path_value)
    return path if path.is_absolute() else (REPO_ROOT / path).resolve()


def run_sql(query: str, params: list[Any] | None = None) -> pd.DataFrame:
    with duckdb.connect(str(DB_PATH), read_only=True) as con:
        return con.execute(query, params or []).df()


def table_exists(table_name: str) -> bool:
    return not run_sql(
        "SELECT 1 FROM information_schema.tables WHERE table_schema = 'main' AND table_name = ? LIMIT 1",
        [table_name],
    ).empty


def table_row_count(table_name: str) -> int | None:
    if not table_exists(table_name):
        return None
    return int(run_sql(f"SELECT COUNT(*) AS row_count FROM {table_name}")["row_count"].iloc[0])


def list_parquet_files(path_value: str) -> list[Path]:
    path = resolve_path(path_value)
    if path.is_file():
        return [path] if path.exists() else []
    if path.is_dir():
        return sorted(p for p in path.glob("*.parquet") if p.is_file())
    return []


def parquet_scan_target(dataset_name: str) -> str:
    dataset_path = resolve_path(DATASET_REGISTRY[dataset_name]["parquet_path"])
    if dataset_path.is_dir():
        return str((dataset_path / "*.parquet").resolve())
    return str(dataset_path.resolve())


def read_parquet_target(target: str | Path, limit: int | None = 20, order_by: str | None = None) -> pd.DataFrame:
    target = str(target).replace("'", "''")
    query = f"SELECT * FROM read_parquet('{target}')"
    if order_by:
        query += f" ORDER BY {order_by}"
    if limit is not None:
        query += f" LIMIT {int(limit)}"
    return run_sql(query)


def dataset_files_df(dataset_name: str) -> pd.DataFrame:
    rows = []
    for path in list_parquet_files(DATASET_REGISTRY[dataset_name]["parquet_path"]):
        rows.append({
            "file_name": path.name,
            "file_path": str(path),
            "size_bytes": path.stat().st_size,
        })
    return pd.DataFrame(rows)


def symbol_parquet_file(dataset_name: str, symbol: str) -> Path:
    symbol = str(symbol).upper()
    files = [path for path in list_parquet_files(DATASET_REGISTRY[dataset_name]["parquet_path"]) if path.name.startswith(f"{symbol}_")]
    if not files:
        raise FileNotFoundError(f"No parquet file found for dataset={dataset_name}, symbol={symbol}")
    return sorted(files)[0]


def file_symbol_set(dataset_name: str) -> set[str]:
    files = list_parquet_files(DATASET_REGISTRY[dataset_name]["parquet_path"])
    return {path.name.split("_")[0] for path in files}


## Quick Parquet Reads

Các cell dưới đây dùng để đọc nhanh dữ liệu Parquet ở layer base và metadata.

In [ ]:
daily_files_df = dataset_files_df("daily_ohlcv_10y")
daily_files_df


In [ ]:
daily_symbol = "BID"
daily_file_path = symbol_parquet_file("daily_ohlcv_10y", daily_symbol)
daily_parquet_df = read_parquet_target(daily_file_path, limit=200, order_by="trading_date DESC")
print(daily_file_path)
daily_parquet_df


In [ ]:
intraday_files_df = dataset_files_df("intraday_ohlcv_15m_60d")
intraday_files_df


In [ ]:
intraday_symbol = "BID"
intraday_file_path = symbol_parquet_file("intraday_ohlcv_15m_60d", intraday_symbol)
intraday_parquet_df = read_parquet_target(intraday_file_path, limit=200, order_by="bar_time DESC")
print(intraday_file_path)
intraday_parquet_df


In [ ]:
metadata_file_path = resolve_path(DATASET_REGISTRY["data_file_manifest"]["parquet_path"])
metadata_parquet_df = read_parquet_target(metadata_file_path, limit=200, order_by="dataset_name, symbol")
print(metadata_file_path)
metadata_parquet_df


In [ ]:
inventory_rows = []

for dataset_name, cfg in DATASET_REGISTRY.items():
    parquet_path = resolve_path(cfg["parquet_path"])
    files = list_parquet_files(cfg["parquet_path"])
    duckdb_table = cfg.get("duckdb_table")
    db_rows = table_row_count(duckdb_table) if duckdb_table else None

    if (db_rows or 0) > 0 and len(files) == 0 and cfg.get("format") == "parquet":
        materialization_status = "db_only"
    elif len(files) > 0 and (db_rows is None or db_rows == 0):
        materialization_status = "files_only"
    elif len(files) > 0 or (db_rows is not None and db_rows > 0):
        materialization_status = "materialized"
    else:
        materialization_status = "not_materialized"

    inventory_rows.append(
        {
            "dataset_name": dataset_name,
            "layer": cfg.get("layer"),
            "duckdb_table": duckdb_table,
            "db_rows": db_rows,
            "file_count": len(files),
            "path_exists": parquet_path.exists(),
            "materialization_status": materialization_status,
            "parquet_path": str(parquet_path.relative_to(REPO_ROOT)),
        }
    )

inventory_df = pd.DataFrame(inventory_rows).sort_values(["layer", "dataset_name"]).reset_index(drop=True)
inventory_df


In [ ]:
active_universe_df = run_sql(
    """
    SELECT DISTINCT symbol
    FROM universe_members
    WHERE universe_name = 'VN30'
      AND (end_date IS NULL OR end_date >= CURRENT_DATE)
    ORDER BY symbol
    """
)

expected_symbols = set(active_universe_df["symbol"].astype(str).tolist())

current_phase_summary_df = pd.DataFrame(
    [
        {
            "dataset_name": "universe_members",
            "expected_items": len(expected_symbols),
            "db_items": int(run_sql("SELECT COUNT(DISTINCT symbol) AS n FROM universe_members")["n"].iloc[0]),
            "file_count": len(list_parquet_files(DATASET_REGISTRY["universe_members"]["parquet_path"])),
            "missing_symbols": 0,
            "extra_symbols": 0,
        },
        {
            "dataset_name": "daily_ohlcv_10y",
            "expected_items": len(expected_symbols),
            "db_items": int(run_sql("SELECT COUNT(DISTINCT symbol) AS n FROM daily_ohlcv_base")["n"].iloc[0]),
            "file_count": len(list_parquet_files(DATASET_REGISTRY["daily_ohlcv_10y"]["parquet_path"])),
            "missing_symbols": len(expected_symbols - file_symbol_set("daily_ohlcv_10y")),
            "extra_symbols": len(file_symbol_set("daily_ohlcv_10y") - expected_symbols),
        },
        {
            "dataset_name": "intraday_ohlcv_15m_60d",
            "expected_items": len(expected_symbols),
            "db_items": int(run_sql("SELECT COUNT(DISTINCT symbol) AS n FROM intraday_ohlcv_15m_base")["n"].iloc[0]),
            "file_count": len(list_parquet_files(DATASET_REGISTRY["intraday_ohlcv_15m_60d"]["parquet_path"])),
            "missing_symbols": len(expected_symbols - file_symbol_set("intraday_ohlcv_15m_60d")),
            "extra_symbols": len(file_symbol_set("intraday_ohlcv_15m_60d") - expected_symbols),
        },
        {
            "dataset_name": "data_file_manifest",
            "expected_items": len(expected_symbols) * 2,
            "db_items": int(
                run_sql(
                    "SELECT COUNT(*) AS n FROM data_file_manifest WHERE dataset_name IN ('daily_ohlcv_10y', 'intraday_ohlcv_15m_60d')"
                )["n"].iloc[0]
            ),
            "file_count": len(list_parquet_files(DATASET_REGISTRY["data_file_manifest"]["parquet_path"])),
            "missing_symbols": 0,
            "extra_symbols": 0,
        },
    ]
)
current_phase_summary_df


In [ ]:
def audit_symbol_files(dataset_name: str, table_name: str, time_column: str) -> pd.DataFrame:
    manifest_df = run_sql(
        """
        SELECT dataset_name, symbol, file_role, file_name, file_path, row_count,
               coverage_start, coverage_end, snapshot_date, update_status
        FROM data_file_manifest
        WHERE dataset_name = ?
        ORDER BY symbol
        """,
        [dataset_name],
    )
    db_df = run_sql(
        f"""
        SELECT symbol,
               COUNT(*) AS db_rows,
               MIN({time_column}) AS db_start,
               MAX({time_column}) AS db_end
        FROM {table_name}
        GROUP BY symbol
        ORDER BY symbol
        """
    )
    db_map = {row["symbol"]: row for _, row in db_df.iterrows()}
    audit_rows = []

    for _, row in manifest_df.iterrows():
        file_path = Path(row["file_path"])
        file_exists = file_path.exists()
        parquet_rows = None
        parquet_start = None
        parquet_end = None
        single_symbol_file = False

        if file_exists:
            frame = pd.read_parquet(file_path)
            parquet_rows = int(len(frame))
            if not frame.empty:
                parquet_start = pd.to_datetime(frame[time_column]).min()
                parquet_end = pd.to_datetime(frame[time_column]).max()
                symbols_in_file = sorted(frame["symbol"].astype(str).unique().tolist())
                single_symbol_file = symbols_in_file == [row["symbol"]]

        db_row = db_map.get(row["symbol"])
        db_rows = int(db_row["db_rows"]) if db_row is not None else None
        db_start = pd.to_datetime(db_row["db_start"]) if db_row is not None else None
        db_end = pd.to_datetime(db_row["db_end"]) if db_row is not None else None
        manifest_start = pd.to_datetime(row["coverage_start"]) if pd.notna(row["coverage_start"]) else None
        manifest_end = pd.to_datetime(row["coverage_end"]) if pd.notna(row["coverage_end"]) else None

        audit_rows.append(
            {
                "dataset_name": dataset_name,
                "symbol": row["symbol"],
                "manifest_status": row["update_status"],
                "file_exists": file_exists,
                "single_symbol_file": single_symbol_file,
                "row_count_match": file_exists and (int(row["row_count"]) == parquet_rows == db_rows),
                "coverage_match": file_exists and (manifest_start == parquet_start == db_start) and (manifest_end == parquet_end == db_end),
                "manifest_rows": int(row["row_count"]),
                "parquet_rows": parquet_rows,
                "db_rows": db_rows,
                "file_name": row["file_name"],
            }
        )

    return pd.DataFrame(audit_rows).sort_values("symbol").reset_index(drop=True)


daily_file_audit_df = audit_symbol_files("daily_ohlcv_10y", "daily_ohlcv_base", "trading_date")
intraday_file_audit_df = audit_symbol_files("intraday_ohlcv_15m_60d", "intraday_ohlcv_15m_base", "bar_time")

file_audit_summary_df = pd.DataFrame(
    [
        {
            "dataset_name": "daily_ohlcv_10y",
            "symbols_checked": len(daily_file_audit_df),
            "missing_files": int((~daily_file_audit_df["file_exists"]).sum()),
            "row_count_mismatches": int((~daily_file_audit_df["row_count_match"]).sum()),
            "coverage_mismatches": int((~daily_file_audit_df["coverage_match"]).sum()),
            "multi_symbol_file_issues": int((~daily_file_audit_df["single_symbol_file"]).sum()),
        },
        {
            "dataset_name": "intraday_ohlcv_15m_60d",
            "symbols_checked": len(intraday_file_audit_df),
            "missing_files": int((~intraday_file_audit_df["file_exists"]).sum()),
            "row_count_mismatches": int((~intraday_file_audit_df["row_count_match"]).sum()),
            "coverage_mismatches": int((~intraday_file_audit_df["coverage_match"]).sum()),
            "multi_symbol_file_issues": int((~intraday_file_audit_df["single_symbol_file"]).sum()),
        },
    ]
)
file_audit_summary_df


In [ ]:
def table_integrity_metrics(table_name: str, time_column: str, intraday: bool = False) -> pd.DataFrame:
    metrics = run_sql(
        f"""
        SELECT
            COUNT(*) AS row_count,
            COUNT(*) - COUNT(DISTINCT symbol || '|' || CAST({time_column} AS VARCHAR)) AS duplicate_keys,
            SUM(CASE WHEN open IS NULL OR high IS NULL OR low IS NULL OR close IS NULL OR volume IS NULL THEN 1 ELSE 0 END) AS null_required_fields,
            SUM(CASE WHEN open <= 0 OR high <= 0 OR low <= 0 OR close <= 0 THEN 1 ELSE 0 END) AS non_positive_prices,
            SUM(CASE WHEN volume < 0 THEN 1 ELSE 0 END) AS negative_volume,
            SUM(CASE WHEN high < low THEN 1 ELSE 0 END) AS high_below_low,
            SUM(CASE WHEN high < GREATEST(open, close, low) THEN 1 ELSE 0 END) AS high_out_of_bounds,
            SUM(CASE WHEN low > LEAST(open, close, high) THEN 1 ELSE 0 END) AS low_out_of_bounds
        FROM {table_name}
        """
    )
    if intraday:
        intraday_extra = run_sql(
            f"""
            SELECT
                SUM(CASE WHEN EXTRACT(minute FROM {time_column}) NOT IN (0, 15, 30, 45) THEN 1 ELSE 0 END) AS invalid_interval_minutes,
                SUM(CASE WHEN session_date <> CAST({time_column} AS DATE) THEN 1 ELSE 0 END) AS session_date_mismatch
            FROM {table_name}
            """
        )
        metrics["invalid_interval_minutes"] = intraday_extra["invalid_interval_minutes"]
        metrics["session_date_mismatch"] = intraday_extra["session_date_mismatch"]
    metrics.insert(0, "table_name", table_name)
    return metrics


daily_metrics_df = table_integrity_metrics("daily_ohlcv_base", "trading_date")
intraday_metrics_df = table_integrity_metrics("intraday_ohlcv_15m_base", "bar_time", intraday=True)

pd.concat([daily_metrics_df, intraday_metrics_df], ignore_index=True)


In [ ]:
issues = []

active_datasets = {"universe_members", "daily_ohlcv_10y", "intraday_ohlcv_15m_60d", "data_file_manifest"}

for _, row in inventory_df.iterrows():
    if row["dataset_name"] in active_datasets and row["materialization_status"] != "materialized":
        issues.append(
            {
                "severity": "critical",
                "check": "materialization",
                "dataset_name": row["dataset_name"],
                "detail": f"materialization_status={row['materialization_status']}",
            }
        )
    elif row["dataset_name"] not in active_datasets and row["materialization_status"] == "not_materialized":
        issues.append(
            {
                "severity": "info",
                "check": "future_dataset",
                "dataset_name": row["dataset_name"],
                "detail": "registered but not built yet",
            }
        )

for dataset_name, frame in [("daily_ohlcv_10y", daily_file_audit_df), ("intraday_ohlcv_15m_60d", intraday_file_audit_df)]:
    failing = frame.loc[~(frame["file_exists"] & frame["single_symbol_file"] & frame["row_count_match"] & frame["coverage_match"])]
    if not failing.empty:
        issues.append(
            {
                "severity": "critical",
                "check": "file_manifest_consistency",
                "dataset_name": dataset_name,
                "detail": f"{len(failing)} symbol files failed consistency checks",
            }
        )

metrics_df = pd.concat([daily_metrics_df, intraday_metrics_df], ignore_index=True)
for _, row in metrics_df.iterrows():
    bad_cols = []
    for column in row.index:
        if column in {"table_name", "row_count"}:
            continue
        value = row[column]
        if pd.isna(value):
            continue
        numeric_value = int(value)
        if numeric_value > 0:
            bad_cols.append(f"{column}={numeric_value}")
    if bad_cols:
        issues.append(
            {
                "severity": "critical",
                "check": "table_integrity",
                "dataset_name": row["table_name"],
                "detail": ", ".join(bad_cols),
            }
        )

issues_df = pd.DataFrame(issues)
if issues_df.empty:
    issues_df = pd.DataFrame(
        [
            {
                "severity": "ok",
                "check": "summary",
                "dataset_name": "current_phase",
                "detail": "All active datasets passed the configured integrity checks.",
            }
        ]
    )

issues_df
